# Colab 02 - Pipeline do artigo

Este notebook foi reorganizado a partir da secao `PIPELINE MESHI` do `TPF_INF_493 (1).ipynb`. Ele usa apenas codigo embutido do notebook original, sem chamadas a scripts externos.


## 1. Instalacao e imports


In [ ]:
!pip -q install numpy pandas scipy scikit-learn matplotlib seaborn pyarrow


In [ ]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    adjusted_rand_score,
    adjusted_mutual_info_score,
    v_measure_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix,
)

from google.colab import drive
drive.mount('/content/drive')

RSEED = 42
np.random.seed(RSEED)
sns.set_theme(style='whitegrid', context='notebook')


## 2. Caminhos e parametros


In [ ]:
DATASET_PATH = Path('/content/drive/MyDrive/SBCUP/Dataset/train_test_network.parquet')
ARTEFATO_PATH = Path('/content/drive/MyDrive/SBCUP/Artefato Intermediário/train_test_network_with_clusters_k30.parquet')
RESULTADOS_DIR = Path('/content/drive/MyDrive/SBCUP/Resultados')
GRAFICOS_DIR = Path('/content/drive/MyDrive/SBCUP/Gráficos')

RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)
GRAFICOS_DIR.mkdir(parents=True, exist_ok=True)
ARTEFATO_PATH.parent.mkdir(parents=True, exist_ok=True)

K_CLUSTERS = 30
SVD_COMPONENTS = 50
N_TREES = 150

# O notebook original fazia CV com 10 folds. Deixe True se quiser repetir essa etapa longa.
RODAR_CV = False
CV_N_SPLITS = 10

AVALIACAO_SPLIT = 'test'

def carregar_dataset(path):
    path = Path(path)
    if path.suffix.lower() == '.parquet':
        return pd.read_parquet(path)
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path)
    raise ValueError(f'Formato nao suportado: {path.suffix}')

def salvar_fig(nome, dpi=220):
    caminho = GRAFICOS_DIR / nome
    plt.savefig(caminho, dpi=dpi, bbox_inches='tight')
    print('Figura salva em:', caminho)
    return caminho

print('Dataset:', DATASET_PATH)
print('Artefato intermediario:', ARTEFATO_PATH)
print('Resultados:', RESULTADOS_DIR)
print('Graficos:', GRAFICOS_DIR)


## 3. Leitura, grupos e split TRAIN / VALID / TEST


In [ ]:
df = carregar_dataset(DATASET_PATH)
print('df:', df.shape)
display(df.head())

assert {'label', 'type'}.issubset(df.columns), 'O dataset precisa conter label e type.'
y_bin = df['label'].copy()
y_type = df['type'].copy()

group_parts = []
for col in ['src_ip', 'dst_ip', 'service', 'proto']:
    if col in df.columns:
        group_parts.append(df[col].astype(str).fillna('NA'))

if group_parts:
    groups = group_parts[0]
    for p in group_parts[1:]:
        groups = groups + '|' + p
else:
    groups = pd.Series(np.arange(len(df)).astype(str), index=df.index)

DROP = ['label', 'type', 'split', 'cluster_kmeans30', 'cluster_name', 'purity_test']
X_raw = df.drop(columns=DROP, errors='ignore').copy()

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(sgkf.split(X_raw, y_type, groups=groups))
trva_idx, te_idx = folds[0]

sgkf2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=123)
tr_idx_rel, va_idx_rel = next(
    sgkf2.split(X_raw.iloc[trva_idx], y_type.iloc[trva_idx], groups=groups.iloc[trva_idx])
)
tr_idx = np.array(trva_idx)[tr_idx_rel]
va_idx = np.array(trva_idx)[va_idx_rel]

print('sizes train/valid/test:', len(tr_idx), len(va_idx), len(te_idx))
print('types:', y_type.nunique(), 'labels:', y_bin.nunique())


## 4. Parte nao-supervisionada: featurizacao, SVD e KMeans K30


In [ ]:
class PortsConnectivityFeaturizer(BaseEstimator, TransformerMixin):
    def __init__(self, topN=20, log_counts=True, drop_raw_ips=True):
        self.topN = topN
        self.log_counts = log_counts
        self.drop_raw_ips = drop_raw_ips

    def fit(self, X, y=None):
        X = X.copy()
        X['src_port'] = pd.to_numeric(X['src_port'], errors='coerce')
        X['dst_port'] = pd.to_numeric(X['dst_port'], errors='coerce')
        self.top_src_ = set(X['src_port'].value_counts().head(self.topN).index)
        self.top_dst_ = set(X['dst_port'].value_counts().head(self.topN).index)

        X['src_ip'] = X['src_ip'].astype(str).fillna('NA')
        X['dst_ip'] = X['dst_ip'].astype(str).fillna('NA')
        pair = X['src_ip'] + '->' + X['dst_ip']
        self.vc_pair_ = pair.value_counts()
        self.vc_src_ = X['src_ip'].value_counts()
        self.vc_dst_ = X['dst_ip'].value_counts()
        self.src_unique_dst_ = X.groupby('src_ip')['dst_ip'].nunique()
        self.dst_unique_src_ = X.groupby('dst_ip')['src_ip'].nunique()
        return self

    def transform(self, X):
        X = X.copy()
        X['src_port'] = pd.to_numeric(X['src_port'], errors='coerce')
        X['dst_port'] = pd.to_numeric(X['dst_port'], errors='coerce')
        X['src_port_wk'] = (X['src_port'] < 1024).astype('int8')
        X['dst_port_wk'] = (X['dst_port'] < 1024).astype('int8')
        X['src_port_top'] = X['src_port'].where(X['src_port'].isin(self.top_src_), other='other').astype(str)
        X['dst_port_top'] = X['dst_port'].where(X['dst_port'].isin(self.top_dst_), other='other').astype(str)
        X = X.drop(columns=['src_port', 'dst_port'])

        X['src_ip'] = X['src_ip'].astype(str).fillna('NA')
        X['dst_ip'] = X['dst_ip'].astype(str).fillna('NA')
        pair = X['src_ip'] + '->' + X['dst_ip']
        X['pair_count'] = pair.map(self.vc_pair_).fillna(0).astype(float)
        X['src_out_count'] = X['src_ip'].map(self.vc_src_).fillna(0).astype(float)
        X['dst_in_count'] = X['dst_ip'].map(self.vc_dst_).fillna(0).astype(float)
        X['src_unique_dst'] = X['src_ip'].map(self.src_unique_dst_).fillna(0).astype(float)
        X['dst_unique_src'] = X['dst_ip'].map(self.dst_unique_src_).fillna(0).astype(float)

        if self.log_counts:
            for c in ['pair_count', 'src_out_count', 'dst_in_count', 'src_unique_dst', 'dst_unique_src']:
                X[c] = np.log1p(X[c])

        if self.drop_raw_ips:
            X = X.drop(columns=['src_ip', 'dst_ip'], errors='ignore')
        return X


In [ ]:
X_tr_raw = X_raw.iloc[tr_idx].copy()
X_va_raw = X_raw.iloc[va_idx].copy()
X_te_raw = X_raw.iloc[te_idx].copy()

feat = PortsConnectivityFeaturizer(topN=20, log_counts=True, drop_raw_ips=True)
X_tr_unsup = feat.fit_transform(X_tr_raw)
X_va_unsup = feat.transform(X_va_raw)
X_te_unsup = feat.transform(X_te_raw)

num_cols = X_tr_unsup.select_dtypes(exclude=['object']).columns.tolist()
cat_cols = X_tr_unsup.select_dtypes(include=['object']).columns.tolist()

already_logged = {'pair_count', 'src_out_count', 'dst_in_count', 'src_unique_dst', 'dst_unique_src'}
skew_cols = [c for c in num_cols if c not in ['src_port_wk', 'dst_port_wk'] and c not in already_logged]
flag_cols = [c for c in num_cols if c not in skew_cols]

preprocess_unsup = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
            ('scaler', StandardScaler(with_mean=False)),
        ]), skew_cols),
        ('num_flags', StandardScaler(with_mean=False), flag_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=10), cat_cols),
    ],
    remainder='drop',
)

embedder = Pipeline([
    ('preprocess', preprocess_unsup),
    ('svd', TruncatedSVD(n_components=SVD_COMPONENTS, random_state=RSEED)),
    ('scaler', StandardScaler()),
])

kmeans30 = MiniBatchKMeans(n_clusters=K_CLUSTERS, random_state=RSEED, batch_size=4096, n_init=10)
Z_tr = embedder.fit_transform(X_tr_unsup)
kmeans30.fit(Z_tr)

cl_tr = kmeans30.labels_
cl_va = kmeans30.predict(embedder.transform(X_va_unsup))
cl_te = kmeans30.predict(embedder.transform(X_te_unsup))

df['split'] = 'none'
df.iloc[tr_idx, df.columns.get_loc('split')] = 'train'
df.iloc[va_idx, df.columns.get_loc('split')] = 'valid'
df.iloc[te_idx, df.columns.get_loc('split')] = 'test'

df['cluster_kmeans30'] = -1
df.iloc[tr_idx, df.columns.get_loc('cluster_kmeans30')] = cl_tr
df.iloc[va_idx, df.columns.get_loc('cluster_kmeans30')] = cl_va
df.iloc[te_idx, df.columns.get_loc('cluster_kmeans30')] = cl_te

display(df['cluster_kmeans30'].value_counts().sort_index())


In [ ]:
def metricas_externas_split(nome, idx):
    cl = df.iloc[idx]['cluster_kmeans30'].values
    yt = df.iloc[idx]['type'].values
    yb = df.iloc[idx]['label'].values
    return [
        {
            'split': nome,
            'target': 'type',
            'ARI': adjusted_rand_score(yt, cl),
            'AMI': adjusted_mutual_info_score(yt, cl, average_method='arithmetic'),
            'V': v_measure_score(yt, cl),
        },
        {
            'split': nome,
            'target': 'label',
            'ARI': adjusted_rand_score(yb, cl),
            'AMI': adjusted_mutual_info_score(yb, cl, average_method='arithmetic'),
            'V': v_measure_score(yb, cl),
        },
    ]

metricas_clusters = pd.DataFrame(
    metricas_externas_split('train', tr_idx)
    + metricas_externas_split('valid', va_idx)
    + metricas_externas_split('test', te_idx)
)
metricas_clusters.to_csv(RESULTADOS_DIR / 'metricas_externas_clusters_k30.csv', index=False)
display(metricas_clusters)

df.to_parquet(ARTEFATO_PATH, index=False)
print('Artefato salvo em:', ARTEFATO_PATH)


## 5. Graficos e tabelas dos clusters K30


In [ ]:
df_eval = df[df['split'].eq(AVALIACAO_SPLIT)].copy()
print('Split de avaliacao:', AVALIACAO_SPLIT, df_eval.shape)

ct_type = pd.crosstab(df_eval['cluster_kmeans30'], df_eval['type'])
ct_type_norm = ct_type.div(ct_type.sum(axis=1), axis=0).fillna(0)
ct_type.to_csv(RESULTADOS_DIR / 'kmeans30_type_por_cluster_contagens.csv')
ct_type_norm.to_csv(RESULTADOS_DIR / 'kmeans30_type_por_cluster_normalizado.csv')

plt.figure(figsize=(11, 7))
sns.heatmap(ct_type_norm, cmap='viridis', annot=False, cbar_kws={'label': 'Proporcao no cluster'})
plt.xlabel('type')
plt.ylabel('cluster_kmeans30')
plt.title(f'Heatmap type por cluster K30 ({AVALIACAO_SPLIT})')
plt.tight_layout()
salvar_fig('kmeans30_heatmap_type_por_cluster.png')
plt.show()

plt.figure(figsize=(11, 7))
sns.heatmap(ct_type, cmap='mako', annot=True, fmt='d', cbar_kws={'label': 'Contagem'})
plt.xlabel('type')
plt.ylabel('cluster_kmeans30')
plt.title(f'Matriz cluster K30 x type - contagens ({AVALIACAO_SPLIT})')
plt.tight_layout()
salvar_fig('kmeans30_matriz_cluster_type_contagens.png')
plt.show()


In [ ]:
cluster_sizes_eval = ct_type.sum(axis=1)
cluster_max = ct_type.max(axis=1)
purity_type = (cluster_max / cluster_sizes_eval).fillna(0)

pureza_df = pd.DataFrame({
    'cluster': purity_type.index,
    'purity_type': purity_type.values,
    'size': cluster_sizes_eval.values,
    'dominant_type': ct_type.idxmax(axis=1).values,
}).sort_values(['purity_type', 'size'], ascending=[False, False])
pureza_df.to_csv(RESULTADOS_DIR / 'kmeans30_pureza_por_cluster.csv', index=False)
display(pureza_df.head(15))

plt.figure(figsize=(11, 4))
sns.barplot(data=pureza_df.sort_values('cluster'), x='cluster', y='purity_type', color='C0')
plt.axhline(0.90, color='red', linestyle='--', label='0.90')
plt.axhline(0.95, color='green', linestyle='--', label='0.95')
plt.xlabel('cluster_kmeans30')
plt.ylabel('pureza em type')
plt.title(f'Pureza por cluster ({AVALIACAO_SPLIT})')
plt.legend()
plt.tight_layout()
salvar_fig('kmeans30_pureza_por_cluster.png')
plt.show()


In [ ]:
thresholds = np.linspace(0.50, 1.00, 51)
total_samples = cluster_sizes_eval.sum()
coverage = []
for t in thresholds:
    covered = cluster_sizes_eval[purity_type >= t].sum() / total_samples
    coverage.append(float(covered))

coverage_df = pd.DataFrame({'purity_threshold': thresholds, 'coverage': coverage})
coverage_df.to_csv(RESULTADOS_DIR / 'kmeans30_cobertura_vs_limiar_pureza.csv', index=False)

plt.figure(figsize=(7, 4))
plt.plot(thresholds, coverage, marker='o')
plt.xlabel('Limiar de pureza')
plt.ylabel('Cobertura das amostras')
plt.title(f'Cobertura vs limiar de pureza ({AVALIACAO_SPLIT})')
plt.grid(True, alpha=0.3)
plt.tight_layout()
salvar_fig('kmeans30_cobertura_vs_limiar_pureza.png')
plt.show()

for t in [0.90, 0.95]:
    cov_t = cluster_sizes_eval[purity_type >= t].sum() / total_samples
    print(f'Cobertura para pureza >= {t:.2f}: {100 * cov_t:.2f}%')


In [ ]:
cluster_sizes_all = df['cluster_kmeans30'].value_counts().sort_index()
cluster_sizes_by_split = pd.crosstab(df['cluster_kmeans30'], df['split'])
cluster_sizes_all.to_csv(RESULTADOS_DIR / 'kmeans30_tamanho_clusters.csv', header=['size'])
cluster_sizes_by_split.to_csv(RESULTADOS_DIR / 'kmeans30_tamanho_clusters_por_split.csv')

plt.figure(figsize=(11, 4))
sns.barplot(x=cluster_sizes_all.index.astype(str), y=cluster_sizes_all.values, color='C0')
plt.xlabel('cluster_kmeans30')
plt.ylabel('numero de amostras')
plt.title('Distribuicao de tamanho dos clusters K30')
plt.tight_layout()
salvar_fig('kmeans30_distribuicao_tamanho_clusters.png')
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(cluster_sizes_all.values, bins=15, color='C0', edgecolor='black')
plt.xlabel('tamanho do cluster')
plt.ylabel('frequencia')
plt.title('Histograma do tamanho dos clusters K30')
plt.tight_layout()
salvar_fig('kmeans30_histograma_tamanho_clusters.png')
plt.show()


## 6. Type estimado pelo tipo dominante do cluster


In [ ]:
df_map = df[df['split'].isin(['train', 'valid'])].copy()
df_test = df[df['split'].eq('test')].copy()

ct_map = pd.crosstab(df_map['cluster_kmeans30'], df_map['type'])
cluster_to_type = ct_map.idxmax(axis=1).to_dict()

y_true_type_cluster = df_test['type'].astype(str)
y_pred_type_cluster = df_test['cluster_kmeans30'].map(cluster_to_type).fillna('unknown').astype(str)

mcc_cluster_type = matthews_corrcoef(y_true_type_cluster, y_pred_type_cluster)
print('MCC type por tipo dominante do cluster:', mcc_cluster_type)
print(classification_report(y_true_type_cluster, y_pred_type_cluster, digits=4, zero_division=0))

report_cluster_type = pd.DataFrame(
    classification_report(y_true_type_cluster, y_pred_type_cluster, output_dict=True, zero_division=0)
).T
report_cluster_type.to_csv(RESULTADOS_DIR / 'metricas_type_por_cluster_dominante.csv')

labels_type = sorted(pd.unique(pd.concat([y_true_type_cluster, y_pred_type_cluster])), key=str)
cm = confusion_matrix(y_true_type_cluster, y_pred_type_cluster, labels=labels_type)
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f', xticklabels=labels_type, yticklabels=labels_type, cmap='Blues')
plt.xlabel('type predito pelo cluster dominante')
plt.ylabel('type verdadeiro')
plt.title('Matriz de confusao - type por cluster dominante (TEST)')
plt.tight_layout()
salvar_fig('kmeans30_matriz_confusao_type_dominante.png')
plt.show()


## 7. Parte supervisionada do pipeline


In [ ]:
leak_cols = ['label', 'type', 'cluster_kmeans30', 'cluster_name', 'purity_test', 'split']
feature_cols = [c for c in df.columns if c not in leak_cols]

X_all = df[feature_cols].copy()
y_cluster = df['cluster_kmeans30'].astype(int).copy()
y_type_sup = df['type'].astype(str).copy()
y_bin_sup = df['label'].astype(str).copy()

X_tr = X_all.iloc[tr_idx].copy()
X_va = X_all.iloc[va_idx].copy()
X_te = X_all.iloc[te_idx].copy()

yc_tr, yc_va, yc_te = y_cluster.iloc[tr_idx], y_cluster.iloc[va_idx], y_cluster.iloc[te_idx]
yt_tr, yt_va, yt_te = y_type_sup.iloc[tr_idx], y_type_sup.iloc[va_idx], y_type_sup.iloc[te_idx]
yb_tr, yb_va, yb_te = y_bin_sup.iloc[tr_idx], y_bin_sup.iloc[va_idx], y_bin_sup.iloc[te_idx]

num_cols = X_tr.select_dtypes(exclude=['object']).columns.tolist()
cat_cols = X_tr.select_dtypes(include=['object']).columns.tolist()

preprocess_sup = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', min_frequency=10)),
        ]), cat_cols),
    ],
    remainder='drop',
)

print('X train/valid/test:', X_tr.shape, X_va.shape, X_te.shape)
print('Targets cluster/type/label:', yc_tr.nunique(), yt_tr.nunique(), yb_tr.nunique())


In [ ]:
def labels_ordenados(y_true, y_pred):
    return sorted(pd.unique(pd.concat([pd.Series(y_true), pd.Series(y_pred)])), key=lambda x: str(x))


def avaliar_split(nome, split, y_true, y_pred, figsize_cm=(8, 6), gerar_graficos=False):
    metricas = {
        'target': nome,
        'split': split,
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macroF1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weightedF1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'MCC': matthews_corrcoef(y_true, y_pred),
    }

    print(f'\n=== {nome} ({split.upper()}) ===')
    print(metricas)
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    report_df = pd.DataFrame(
        classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    ).T
    report_df.to_csv(RESULTADOS_DIR / f'metricas_por_classe_{nome}_{split}.csv')

    labels = labels_ordenados(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    pd.DataFrame(cm, index=labels, columns=labels).to_csv(
        RESULTADOS_DIR / f'matriz_confusao_{nome}_{split}_contagens.csv'
    )
    pd.DataFrame(cm_norm, index=labels, columns=labels).to_csv(
        RESULTADOS_DIR / f'matriz_confusao_{nome}_{split}_normalizada.csv'
    )

    if gerar_graficos:
        plt.figure(figsize=figsize_cm)
        sns.heatmap(cm_norm, annot=True, fmt='.2f', xticklabels=labels, yticklabels=labels, cmap='Blues')
        plt.xlabel(f'Predito ({nome})')
        plt.ylabel(f'Verdadeiro ({nome})')
        plt.title(f'Matriz de confusao normalizada - {nome} ({split.upper()})')
        plt.tight_layout()
        salvar_fig(f'matriz_confusao_{nome}_{split}.png')
        plt.show()

        per_class = report_df.drop(index=['accuracy', 'macro avg', 'weighted avg'], errors='ignore').reset_index(names='classe')
        per_class = per_class.sort_values('f1-score', ascending=False)
        plt.figure(figsize=(max(7, 0.35 * len(per_class)), 4))
        sns.barplot(data=per_class, x='classe', y='f1-score', color='C0')
        plt.xticks(rotation=45, ha='right')
        plt.ylim(0, 1.0)
        plt.title(f'F1-score por classe - {nome} ({split.upper()})')
        plt.tight_layout()
        salvar_fig(f'f1_por_classe_{nome}_{split}.png')
        plt.show()

    return metricas, report_df


def avaliar_modelo(nome, y_tr, y_va, y_te, figsize_cm=(8, 6)):
    clf = RandomForestClassifier(
        n_estimators=N_TREES,
        random_state=10000,
        n_jobs=-1,
        class_weight='balanced_subsample',
    )
    pipe = Pipeline([('prep', preprocess_sup), ('clf', clf)])

    if RODAR_CV:
        cv = StratifiedGroupKFold(n_splits=CV_N_SPLITS, shuffle=True, random_state=10000)
        scores = cross_val_score(
            pipe, X_tr, y_tr, cv=cv, groups=groups.iloc[tr_idx],
            scoring='f1_macro', n_jobs=-1
        )
        print(f'{nome} - CV macroF1 no TRAIN:', scores.mean(), '+/-', scores.std())

    pipe.fit(X_tr, y_tr)

    # Primeiro VALID: usado como avaliacao intermediaria, antes de olhar o TEST final.
    y_pred_va = pipe.predict(X_va)
    met_va, report_va = avaliar_split(nome, 'valid', y_va, y_pred_va, figsize_cm=figsize_cm, gerar_graficos=False)

    # Depois TEST: avaliacao final, condizente com o protocolo do artigo.
    y_pred_te = pipe.predict(X_te)
    met_te, report_te = avaliar_split(nome, 'test', y_te, y_pred_te, figsize_cm=figsize_cm, gerar_graficos=True)

    return pipe, {'valid': y_pred_va, 'test': y_pred_te}, {'valid': met_va, 'test': met_te}, {'valid': report_va, 'test': report_te}


pipe_cluster, preds_cluster, mets_cluster, reports_cluster = avaliar_modelo(
    'cluster_kmeans30', yc_tr, yc_va, yc_te, figsize_cm=(12, 10)
)
pipe_type, preds_type, mets_type, reports_type = avaliar_modelo(
    'type', yt_tr, yt_va, yt_te, figsize_cm=(10, 8)
)
pipe_label, preds_label, mets_label, reports_label = avaliar_modelo(
    'label', yb_tr, yb_va, yb_te, figsize_cm=(6, 5)
)

# Mantem nomes convenientes para uso posterior, se necessario.
pred_cluster_te = preds_cluster['test']
pred_type_te = preds_type['test']
pred_label_te = preds_label['test']
met_cluster = mets_cluster['test']
met_type = mets_type['test']
met_label = mets_label['test']
report_cluster = reports_cluster['test']
report_type = reports_type['test']
report_label = reports_label['test']

metricas_globais = pd.DataFrame([
    mets_cluster['valid'], mets_cluster['test'],
    mets_type['valid'], mets_type['test'],
    mets_label['valid'], mets_label['test'],
])
metricas_globais.to_csv(RESULTADOS_DIR / 'metricas_globais_supervisionadas.csv', index=False)
display(metricas_globais)


## 8. Arquivos gerados


In [ ]:
print('Resultados CSV:')
for p in sorted(RESULTADOS_DIR.glob('*.csv')):
    print('-', p)

print('\nGraficos PNG:')
for p in sorted(GRAFICOS_DIR.glob('*.png')):
    if any(tag in p.name for tag in ['kmeans30', 'matriz_confusao', 'f1_por_classe']):
        print('-', p)
